# SUPP-C: CDS Integrity Concordance Matrices

Three concordance heatmaps (start codon, stop codon, reading frame) plus
a summary bar showing full/partial/no concordance.

**Input:** `intermediate_spreadsheets/coding_integrity/` from workflow

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

RESULTS_DIR = Path('../results')
CI_DIR = RESULTS_DIR / 'intermediate_spreadsheets' / 'coding_integrity'
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# Load pre-computed data
matrices = pd.read_csv(CI_DIR / 'cds_concordance_matrices.tsv', sep='\t')
per_asm = pd.read_csv(CI_DIR / 'coding_integrity_per_assembly.tsv', sep='\t')
classifications = pd.read_csv(CI_DIR / 'cds_classification_distribution.tsv', sep='\t')

print(f"Per-assembly data: {len(per_asm)} assemblies")
print(f"\nClassification distribution:")
display(classifications)
print(f"\nConcordance matrices:")
display(matrices)

In [ ]:
# --- SUPP-C Figure ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Panel A-C: Per-assembly concordance violin plots for start, stop, frame
metrics = [
    ('pct_start_match', 'A. Start codon concordance', '#3498db'),
    ('pct_stop_match', 'B. Stop codon concordance', '#2ecc71'),
    ('pct_frame_intact', 'C. Frame preservation', '#e74c3c'),
]

for ax, (col, title, color) in zip(axes.flat[:3], metrics):
    data = per_asm[col].dropna()
    parts = ax.violinplot([data], showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    
    med = data.median()
    ax.set_title(f'{title}\nMedian: {med:.1f}%', fontsize=11, fontweight='bold')
    ax.set_ylabel('Concordance (%)', fontsize=10)
    ax.set_xticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Panel D: Classification distribution bar chart
ax = axes[1, 1]
top_n = 8
top_cls = classifications.head(top_n)
colors = plt.cm.Set2(np.linspace(0, 1, top_n))

bars = ax.barh(range(len(top_cls)-1, -1, -1), top_cls['count'], color=colors)
ax.set_yticks(range(len(top_cls)-1, -1, -1))
ax.set_yticklabels(top_cls['classification'], fontsize=9)
ax.set_xlabel('Count (gene-assembly instances)', fontsize=10)
ax.set_title('D. CDS classification distribution', fontsize=11, fontweight='bold')

# Add count labels
for i, (bar, count) in enumerate(zip(bars, top_cls['count'])):
    ax.text(bar.get_width() + max(top_cls['count']) * 0.02, bar.get_y() + bar.get_height()/2,
            f'{count:,}', va='center', fontsize=8)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_supp_c_cds_heatmaps.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figure_supp_c_cds_heatmaps.pdf', bbox_inches='tight')
plt.show()
print(f"Saved to {OUTPUT_DIR / 'figure_supp_c_cds_heatmaps.png'}")

In [ ]:
# Summary bar: proportion fully concordant / partially / fully discordant
fig, ax = plt.subplots(figsize=(10, 2))

# All three properties concordant
n_all_match = per_asm.apply(lambda row: (row['pct_start_match'] == 100) and 
                                         (row['pct_stop_match'] == 100) and 
                                         (row['pct_frame_intact'] == 100), axis=1).sum()

# Compute per-assembly: proportion of pairs where all three match
full_conc_median = per_asm['pct_full_match'].median()
start_only = per_asm['pct_start_match'].median()
stop_only = per_asm['pct_stop_match'].median()
frame_only = per_asm['pct_frame_intact'].median()

print(f"Median % full CDS concordance: {full_conc_median:.1f}%")
print(f"Median start codon concordance: {start_only:.1f}%")
print(f"Median stop codon concordance: {stop_only:.1f}%")
print(f"Median frame preservation: {frame_only:.1f}%")